# Modèle testé sur le dataset medical

Au préalable, importer le fichier "medical_dataset_clean.json" dans Google Collab.


## Librairies



In [1]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes

In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from sklearn.model_selection import train_test_split
from datasets import Dataset

## Dataset

In [3]:
import json

with open("/content/medical_dataset_clean.json", "r", encoding="utf-8") as f:
    data = json.load(f)  # charge tout le array d'un coup

print(f"{len(data)} exemples chargés")
print(data[0])  # vérifier la structure du premier élément

243838 exemples chargés
{'instruction': 'Hi doctor, I am a 22-year-old female who was diagnosed with hypothyroidism (genetic) when I was 12. Over the past five years, I have become around 50 pounds overweight and all of my attempts to lose have seemed to fail so I have given up, but my weight has stayed the same. There is so much information put there about losing weight with hypothyroidism but it all seems to conflict. I am so unsure as to what type of exercise and diet I should follow as a result but I still would like to lose weight, but most importantly have my body feel better. What can I do? I am currently on Levothyroxine, Buspar, and Benedryl.', 'output': 'Hi. You have really done well with the hypothyroidism problem. Your levels are normal with less medications which are very good. As it is genetically induced, it is very difficult to lose weight. My advice to you is, you should focus on maintaining normal levels of TSH (thyroid-stimulating hormone) and try to remain active, h

In [4]:
# Convertir la liste de dictionnaires en DataFrame
df = pd.DataFrame(data)

# Séparer le dataset en ensembles d'entraînement et de test (80/20)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train DataFrame head:")
display(train_df.head())

print("\nTest DataFrame head:")
display(test_df.head())

Train DataFrame head:


,instruction,output
101784,"I have a compression fx at L2, Retrolisthesis ...",hello healthcare user.. i would like you to av...
62761,my sister in law has been feeling fatigue and ...,"Hi, see fatigue is not a big deal but as u sai..."
176430,i am 28 year old male and got married in 2010 ...,"Hai,you have to undergo tubal patency test by ..."
213544,i m 41 years old male with weight 105 kgs.and ...,Hi welcome to health care magic.....Here most ...
20126,"Hi Doctor, I'm 31yrs old & getting married soo...","Hello,Yes it's normal to have a bend in penis ..."



Test DataFrame head:


,instruction,output
122322,my sister has been battling cancer. First she ...,"Hi,Thanks for writing in.Research shows that r..."
143539,14 month old choking last night vomited bits o...,Hi...by what you quote I feel that the kid mig...
224651,"Hi, since last thursday night/early friday mor...","Hello, Chest pain on inspiration is not genera..."
28195,My father in law has dementia and I have been ...,hiThanks for choosing healthcare magicDementia...
18365,hi! my boyfriend and I have been together for ...,Area within foreskin of male penis has secreti...


## Chargement du modèle Phi-3.5 mini 4k avec QLoRA

On va utiliser le modèle `microsoft/Phi-3-mini-4k-instruct` en utilisant la technique QLoRA (Quantized LoRA) pour réduire l'utilisation de la mémoire tout en permettant un fine-tuning efficace.

In [5]:

# Définir l'ID du modèle
model_id = "microsoft/Phi-3-mini-4k-instruct"


In [6]:
# Configuration de la quantisation 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)


In [7]:
from transformers import AutoConfig

# Charger le tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Charger la configuration officielle (trust_remote_code=False par défaut pour utiliser l'intégration transformers)
config = AutoConfig.from_pretrained(model_id, trust_remote_code=False)

# Charger le modèle en utilisant l'implémentation native de transformers
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    config=config,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager"
)

model.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Embedding(32011, 3072, padding_idx=32000)

In [8]:
# Préparer le modèle pour l'entraînement en K-bit
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# Configuration LoRA
lora_config = LoraConfig(
    r=16, # Rang de l'adaptateur LoRA
    lora_alpha=16, # Scaling LoRA alpha
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # Modules à fine-tuner
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Ajouter les adaptateurs LoRA au modèle
model = get_peft_model(model, lora_config)

print("Modèle chargé et configuré avec QLoRA :")
model.print_trainable_parameters()

Modèle chargé et configuré avec QLoRA :
trainable params: 8,912,896 || all params: 3,829,666,816 || trainable%: 0.2327


## Entraînement

In [13]:
from trl import SFTTrainer, SFTConfig
import torch
from datasets import Dataset # Ensure Dataset is imported

# Define input and target columns based on the DataFrame structure
input_col = 'instruction'
target_col = 'output'

# Function to apply ChatML formatting during dataset creation (not passed to SFTTrainer)
def prepare_chat_template(example):
    messages = [
        {"role": "user", "content": example[input_col]},
        {"role": "assistant", "content": example[target_col]},
    ]
    # apply_chat_template returns a string if tokenize=False
    # add_generation_prompt=False because we are providing both user and assistant parts
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

# Convert DataFrames to Hugging Face Datasets
train_dataset_hf = Dataset.from_pandas(train_df[[input_col, target_col]])
eval_dataset_hf = Dataset.from_pandas(test_df[[input_col, target_col]])

# Apply the formatting to create the 'text' column
train_dataset_hf = train_dataset_hf.map(prepare_chat_template, remove_columns=train_dataset_hf.column_names)
eval_dataset_hf = eval_dataset_hf.map(prepare_chat_template, remove_columns=eval_dataset_hf.column_names)

# # Configuration de l'entraînement
# sft_config = SFTConfig(
#     output_dir="./phi-3-qlora-med",
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     logging_steps=5,
#     max_steps=30,
#     save_steps=15,
#     optim="paged_adamw_32bit",
#     bf16=True,
#     lr_scheduler_type="cosine",
#     warmup_steps=5,
#     report_to="none",
#     # dataset_text_field='text' is the default and still applies when using pre-formatted datasets
#     packing=False
# )

sft_config = SFTConfig(
    output_dir="./phi-3-qlora-med",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    max_steps=30,
    save_steps=15,
    optim="paged_adamw_32bit",
    bf16=True,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    report_to="none",
    packing=False
)

# Initialisation du Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_hf, # Pass pre-formatted Hugging Face Dataset
    eval_dataset=eval_dataset_hf,   # Pass pre-formatted Hugging Face Dataset
    args=sft_config,
    # tokenizer=tokenizer, # Removed as per TypeError
)

# Lancement du fine-tuning
print("Début de l'entraînement (30 étapes)...")
trainer.train()

# Sauvegarde finale du modèle et du tokenizer
print("Sauvegarde du modèle...")
trainer.save_model("./phi-3-qlora-med-final")
tokenizer.save_pretrained("./phi-3-qlora-med-final")
print("Entraînement et sauvegarde terminés.")

Map:   0%|          | 0/195070 [00:00<?, ? examples/s]

Map:   0%|          | 0/48768 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/195070 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/195070 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4727 > 4096). Running this sequence through the model will result in indexing errors


...

Step,Training Loss
5,2.798191
10,2.686514
15,2.533601
20,2.476915
25,2.443521
30,2.452086


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Sauvegarde du modèle...


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Entraînement et sauvegarde terminés.
